# Решения: дисбаланс и метрики

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_bank_csv() -> Path:
    for path in (Path("bank_marketing_slim.csv"), Path("../../data/bank_marketing_slim.csv")):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"


CSV_PATH = find_bank_csv()
df = pd.read_csv(CSV_PATH)
target = df["y"].eq("yes").astype(int)
assert len(df) > 0 and set(target.unique()) == {0, 1}
assert "duration" in df.columns  # колонка видна только для разбора утечки
print(f"Строк: {len(df)}; доля yes: {target.mean():.3f}")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


## Урок. 1. Размер классов

In [ ]:
class_counts = target.value_counts().sort_index()
positive_share = float(target.mean())
assert 0 < positive_share < 0.5


## Урок. 2. Baseline

In [ ]:
baseline_pred = np.zeros(len(target), dtype=int)
baseline_accuracy = float(accuracy_score(target, baseline_pred))
baseline_recall = float(recall_score(target, baseline_pred, zero_division=0))
assert baseline_recall == 0.0


## Урок. 3. Модель без утечки

In [ ]:
FEATURE_COLUMNS = [column for column in df.columns if column not in {"y", "duration"}]
assert "duration" not in FEATURE_COLUMNS, "LEAKAGE: duration известна только после звонка"
assert "y" not in FEATURE_COLUMNS
X = pd.get_dummies(df[FEATURE_COLUMNS], drop_first=True)
assert "duration" not in X.columns
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.25, random_state=62, stratify=target
)
model = LogisticRegression(max_iter=1000, solver="liblinear", class_weight="balanced").fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
assert len(proba) == len(y_test)


## Урок. 4. Матрица ошибок

In [ ]:
pred_05 = (proba >= 0.5).astype(int)
tn, fp, fn, tp = (int(x) for x in confusion_matrix(y_test, pred_05).ravel())
assert tn + fp + fn + tp == len(y_test)


## Урок. 5. Метрики вручную

In [ ]:
accuracy_manual = (tn + tp) / (tn + fp + fn + tp)
precision_manual = tp / (tp + fp) if tp + fp else 0.0
recall_manual = tp / (tp + fn) if tp + fn else 0.0
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual) if precision_manual + recall_manual else 0.0
assert all(0 <= x <= 1 for x in (accuracy_manual, precision_manual, recall_manual, f1_manual))


## Урок. 6. Сверка

In [ ]:
sklearn_metrics = {'accuracy': float(accuracy_score(y_test, pred_05)), 'precision': float(precision_score(y_test, pred_05, zero_division=0)), 'recall': float(recall_score(y_test, pred_05, zero_division=0)), 'f1': float(f1_score(y_test, pred_05, zero_division=0))}
assert abs(f1_manual - sklearn_metrics['f1']) < 1e-12


## Урок. 7–8. Цена ошибок

In [ ]:
def metrics_row(y_true, proba_values, threshold):
    pred = (np.asarray(proba_values) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": float(threshold), "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }

ERROR_COSTS = {'fp': 1, 'fn': 5}
cost_05 = ERROR_COSTS['fp'] * fp + ERROR_COSTS['fn'] * fn
cost_rows = []
for threshold in np.arange(0.1, 1.0, 0.1):
    row = metrics_row(y_test, proba, threshold)
    cost_rows.append({'threshold': threshold, 'fp': row['fp'], 'fn': row['fn'], 'cost': row['fp'] + 5 * row['fn']})
cost_table = pd.DataFrame(cost_rows)
best_cost_row = cost_table.loc[cost_table['cost'].idxmin()]
assert len(cost_table) == 9


## Урок. 9. Выбор метрики

In [ ]:
METRIC_NOTE = (f"Baseline получает accuracy={baseline_accuracy:.3f}, но recall=0: он не находит ни одного yes. "
"Поэтому accuracy скрывает пропуски редкого класса. Для кампании смотрим confusion matrix, recall и FN; "
f"при цене FN в пять раз выше FP минимальную стоимость дал порог {best_cost_row.threshold:.1f}. Этот выбор зависит от принятой цены ошибок, а не только от алгоритма.")
assert len(METRIC_NOTE) >= 240


## ДЗ. A1. Pipeline

In [ ]:
assert 'duration' not in FEATURE_COLUMNS and 'duration' not in X.columns


## ДЗ. A2. Функция метрик

In [ ]:
def metrics_row(y_true, proba_values, threshold):
    pred = (np.asarray(proba_values) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": float(threshold), "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }
row = metrics_row(y_test, proba, 0.5)
assert row['tn'] + row['fp'] + row['fn'] + row['tp'] == len(y_test)


## ДЗ. A3. Таблица

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
metric_table = pd.DataFrame([metrics_row(y_test, proba, t) for t in thresholds])
assert len(metric_table) == 17


## ДЗ. Challenge. Сценарии

In [ ]:
cost_scenarios = {'balanced': (1, 1), 'miss_expensive': (1, 6), 'call_expensive': (5, 1)}
choices = []
for name, (fp_cost, fn_cost) in cost_scenarios.items():
    costs = metric_table['fp'] * fp_cost + metric_table['fn'] * fn_cost
    best = metric_table.loc[costs.idxmin()]
    choices.append({'scenario': name, 'fp_cost': fp_cost, 'fn_cost': fn_cost, 'threshold': float(best.threshold), 'total_cost': int(costs.min())})
choice_table = pd.DataFrame(choices)
assert len(choice_table) == 3


## ДЗ. Challenge. Рекомендация

In [ ]:
values = dict(zip(choice_table["scenario"], choice_table["threshold"]))
COST_NOTE = (
    f"balanced: порог {values['balanced']:.2f} при одинаковой цене FP и FN. "
    f"miss_expensive: порог {values['miss_expensive']:.2f}, потому что пропуск yes дороже и нужен больший recall. "
    f"call_expensive: порог {values['call_expensive']:.2f}, потому что лишний звонок дорог и важнее precision. "
    "Различие порогов показывает: оптимального порога вне операционного сценария нет; "
    "цены ошибок нужно согласовать до выбора модели."
)
assert len(COST_NOTE) >= 280
